In [3]:
import boto3
import json

bedrock = boto3.client(service_name='bedrock-runtime')
bedrock_agent = boto3.client(service_name='bedrock-agent')

MODEL_ID = "amazon.nova-micro-v1:0"

### Task 2.2: Traveling with Bedrock

In [4]:
temperature = .7
inference_config = {"temperature": temperature}

system_prompts = [{"text": "You are a virtual travel assistant that suggests destinations based on user preferences."
                + "Only return destination names and a brief description."}]

messages = []

message_1 = {
    "role": "user",
    "content": [{"text": "Create a list of 3 travel destinations."}]
}

messages.append(message_1)

response = bedrock.converse(
    modelId=MODEL_ID,
    messages=messages,
    system=system_prompts,
    inferenceConfig=inference_config
)

def print_response(response):
    model_response = response.get('output', {}).get('message', {}).get('content', [{}])[0].get('text', '')

    print("✈️ Your suggested travel destinations:")
    print(model_response)

print_response(response)

✈️ Your suggested travel destinations:
1. **Kyoto, Japan**
   - Experience traditional Japanese culture, stunning temples, and beautiful gardens.

2. **Santorini, Greece**
   - Enjoy picturesque views, charming white-washed buildings, and delicious local cuisine.

3. **Banff, Canada**
   - Explore stunning mountain landscapes, clear lakes, and outdoor adventure activities.


### Task 2.3: Continuing the conversation

In [5]:
message_2 = {
        "role": "user",
        "content": [{"text": "Only suggest travel locations that are no more than one short flight away."}]
}

messages.append(message_2)

response = bedrock.converse(
    modelId=MODEL_ID,
    messages=messages,
    system=system_prompts,
    inferenceConfig=inference_config
)

print_response(response)

✈️ Your suggested travel destinations:
1. **Orlando, Florida**  
   A family-friendly paradise with world-famous theme parks like Disney World and Universal Studios.

2. **Miami, Florida**  
   A vibrant city known for its beautiful beaches, lively nightlife, and rich cultural experiences.

3. **San Diego, California**  
   A coastal gem offering stunning beaches, renowned zoos, and a lively waterfront district.


### Task 2.4: Your turn to refine the results

In [6]:
message_3 = {
        "role": "user",
        "content": [{"text": "you could use List destinations by their proximity to Seattle, WA USA."}]
}

messages.append(message_3)

response = bedrock.converse(
    modelId=MODEL_ID,
    messages=messages,
    system=system_prompts,
    inferenceConfig=inference_config
)

print_response(response)

✈️ Your suggested travel destinations:
1. **Vancouver, Canada**  
   Just a short ferry or flight ride from Seattle, Vancouver offers stunning natural beauty, vibrant culture, and a lively culinary scene.

2. **Portland, Oregon**  
   About a 2.5-hour drive or a one-hour flight from Seattle, Portland is known for its lush parks, independent breweries, and eclectic food market.

3. **Victoria, Canada**  
   A quick 1.5-hour flight or ferry ride from Seattle, Victoria boasts British Columbia’s rich history, charming Victorian architecture, and beautiful gardens.


### Task 3: Creating a Bedrock Prompt

In [7]:
try:
    response = bedrock_agent.create_prompt(
        name="Travel-Agent-Prompt",
        description="Checks if all trip information has been provided.",
        variants=[
            { 
                "name": "Variant1",
                "modelId": MODEL_ID,
                "templateType": "CHAT",
                "inferenceConfiguration": {
                    "text": {
                        "temperature": 0.4
                    }
                },
                "templateConfiguration": { 
                    "chat": {
                        'system': [ 
                            {
                                "text": """You are a travel agent evaluating trip requests for custom itineraries. 
                                Review the message carefully and answer YES or NO to the following screening questions. 
                                Be strict—if any detail is missing or unclear, answer NO.

                                A) Is the destination clearly stated?
                                B) Are the travel dates within a reasonable range (not last−minute or over a year away)?
                                C) Does the request avoid high−risk or restricted activities (e.g., extreme sports, off−grid travel)?
                                D) Is there any mention of a valid passport or travel documentation?
                                E) Is there enough information to follow up with a proposed itinerary?"""
                            }
                        ],
                        'messages': [{
                            'role': 'user',
                            'content': [ 
                                {
                                    'text': "Trip request: {{event_request}}"
                                }
                            ]
                        }],
                        'inputVariables' : [
                            { 'name' : 'event_request'}
                        ]
                    }
                }
        }]
    )
    print("Created!")
    prompt_arn = response.get("arn")
except bedrock.exceptions.ConflictException as e:
    print("Already exists!")
    response = bedrock.list_prompts()
    prompt = next((prompt for prompt in response['promptSummaries'] if prompt['name'] == "TripBooker_xyz"), None)
    prompt_arn = prompt['arn']

prompt_arn

Created!


'arn:aws:bedrock:us-east-1:938176218293:prompt/TKACD7GNXC'

In [8]:
response = bedrock.converse(
    modelId=prompt_arn,
    promptVariables={
        'event_request': {
            'text': """
                Hi there! I'm planning a trip to Italy with my partner and would love some help organizing the itinerary. We're hoping to travel between September 10–20 this year, ideally flying into Rome and spending a few days in Florence and Venice as well. We’d love recommendations on tours, cultural sites, and good local restaurants. We’re not interested in anything risky like skydiving or hiking remote trails — just want a relaxing and enriching experience. We both have valid passports. Let me know what other details you need!
                """
        }
    },
)
print(response['output']['message']['content'][0]['text'])

A) Is the destination clearly stated?
YES

B) Are the travel dates within a reasonable range (not last−minute or over a year away)?
YES

C) Does the request avoid high−risk or restricted activities (e.g., extreme sports, off−grid travel)?
YES

D) Is there any mention of a valid passport or travel documentation?
YES

E) Is there enough information to follow up with a proposed itinerary?
YES

Answer: YES

The request provides a clear destination, reasonable travel dates, avoids high-risk activities, mentions valid passports, and provides sufficient details for itinerary planning.
